# Atualizar coordenadas da tabela `loja`


In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

username = "root"
password = "pass"
host = "localhost"
port = 3306
database = "DSIA2"

engine = create_engine(f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}")
csv_path = Path.home() / "Documents" / "Cidades.csv"

print(f"CSV: {csv_path}")

CSV: C:\Users\Gonçalo\Documents\Cidades.csv


In [2]:
# Confirmar a ligacao e a existencia da tabela loja.
with engine.connect() as conn:
    db_name = conn.execute(text("SELECT DATABASE()")).scalar()
    loja_exists = conn.execute(text("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = DATABASE()
          AND table_name = 'loja'
    """)).scalar()

print(f"Base de dados ligada: {db_name}")

if not loja_exists:
    raise RuntimeError("A tabela loja nao existe nesta base de dados.")

print("Tabela loja encontrada.")

Base de dados ligada: dsia2
Tabela loja encontrada.


In [3]:
# Adicionar colunas apenas se ainda nao existirem.
columns_to_add = {
    "Latitude": "DECIMAL(10,7) NULL",
    "Longitude": "DECIMAL(10,7) NULL",
}

with engine.begin() as conn:
    existing_columns = set(conn.execute(text("""
        SELECT COLUMN_NAME
        FROM information_schema.columns
        WHERE table_schema = DATABASE()
          AND table_name = 'loja'
    """)).scalars().all())

    for column_name, column_type in columns_to_add.items():
        if column_name in existing_columns:
            print(f"Coluna {column_name} ja existe.")
        else:
            conn.execute(text(f"ALTER TABLE loja ADD COLUMN {column_name} {column_type}"))
            print(f"Coluna {column_name} adicionada.")

Coluna Latitude ja existe.
Coluna Longitude ja existe.


In [4]:
def normalizar_cidade(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

# Estas cidades aparecem com grafia diferente entre a tabela loja e o CSV.
cidade_aliases = {
    "AETHELNEY": "ATHELNEY",
    "PALPERROTH": "PALPERROUTH",
}

if not csv_path.exists():
    raise FileNotFoundError(f"Ficheiro nao encontrado: {csv_path}")

df_cidades = pd.read_csv(csv_path, sep=";", decimal=",", encoding="utf-8-sig")
df_cidades.columns = df_cidades.columns.astype(str).str.strip()

required_columns = {"Cidade", "Lat", "Long"}
missing_columns = required_columns - set(df_cidades.columns)

if missing_columns:
    raise ValueError(f"Colunas em falta no CSV: {sorted(missing_columns)}")

df_cidades = df_cidades.rename(columns={"Lat": "Latitude", "Long": "Longitude"})
df_cidades["Cidade_Normalizada"] = normalizar_cidade(df_cidades["Cidade"])
df_cidades["Latitude"] = pd.to_numeric(df_cidades["Latitude"], errors="coerce")
df_cidades["Longitude"] = pd.to_numeric(df_cidades["Longitude"], errors="coerce")

invalid_rows = df_cidades[
    df_cidades["Cidade_Normalizada"].isna()
    | df_cidades["Latitude"].isna()
    | df_cidades["Longitude"].isna()
    | ~df_cidades["Latitude"].between(-90, 90)
    | ~df_cidades["Longitude"].between(-180, 180)
]

if not invalid_rows.empty:
    display(invalid_rows)
    raise ValueError("Existem linhas invalidas no CSV. Corrige-as antes de atualizar a base de dados.")

duplicated_cities = df_cidades[df_cidades["Cidade_Normalizada"].duplicated(keep=False)]
if not duplicated_cities.empty:
    print("Aviso: existem cidades repetidas no CSV. Sera usada a primeira ocorrencia de cada cidade.")
    display(duplicated_cities.sort_values("Cidade_Normalizada"))

df_cidades = (
    df_cidades
    .drop_duplicates(subset=["Cidade_Normalizada"], keep="first")
    [["Cidade_Normalizada", "Latitude", "Longitude"]]
)

print(f"Cidades validas no CSV: {len(df_cidades)}")
display(df_cidades.head())

Cidades validas no CSV: 68


,Cidade_Normalizada,Latitude,Longitude
0,ATHELNEY,51.05640,-2.93220
1,ALNERWICK,55.41318,-1.70563
2,ARBINGTON,52.24940,0.86460
3,ASHBORNE,53.01670,-1.73330
4,BALERNO,55.88570,-3.34240


In [5]:
# Fazer correspondencia por cidade e atualizar a loja por ID_Loja.
df_loja = pd.read_sql(text("SELECT ID_Loja, N_Loja, Cidade FROM loja"), con=engine)
df_loja["Cidade_Normalizada"] = normalizar_cidade(df_loja["Cidade"])
df_loja["Cidade_Normalizada"] = df_loja["Cidade_Normalizada"].replace(cidade_aliases)

df_update = df_loja.merge(df_cidades, on="Cidade_Normalizada", how="left")

df_sem_coordenadas = df_update[
    df_update["Latitude"].isna() | df_update["Longitude"].isna()
].copy()

df_para_atualizar = df_update.dropna(subset=["Latitude", "Longitude"]).copy()

print(f"Lojas encontradas: {len(df_loja)}")
print(f"Lojas com coordenadas correspondentes: {len(df_para_atualizar)}")
print(f"Lojas sem coordenadas correspondentes: {len(df_sem_coordenadas)}")

if not df_sem_coordenadas.empty:
    display(
        df_sem_coordenadas[["ID_Loja", "N_Loja", "Cidade"]]
        .drop_duplicates()
        .sort_values("Cidade")
    )

df_para_atualizar = df_para_atualizar.astype({
    "ID_Loja": "int64",
    "Latitude": "float64",
    "Longitude": "float64",
})

params = (
    df_para_atualizar[["ID_Loja", "Latitude", "Longitude"]]
    .rename(columns={
        "ID_Loja": "id_loja",
        "Latitude": "latitude",
        "Longitude": "longitude",
    })
    .to_dict("records")
)

if params:
    with engine.begin() as conn:
        conn.execute(text("""
            UPDATE loja
            SET Latitude = :latitude,
                Longitude = :longitude
            WHERE ID_Loja = :id_loja
        """), params)

print(f"Atualizacao concluida para {len(params)} lojas.")

Lojas encontradas: 80
Lojas com coordenadas correspondentes: 80
Lojas sem coordenadas correspondentes: 0
Atualizacao concluida para 80 lojas.


In [6]:
# Validacao final.
resumo = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_lojas,
        SUM(CASE WHEN Latitude IS NOT NULL AND Longitude IS NOT NULL THEN 1 ELSE 0 END) AS lojas_com_coordenadas,
        SUM(CASE WHEN Latitude IS NULL OR Longitude IS NULL THEN 1 ELSE 0 END) AS lojas_sem_coordenadas
    FROM loja
"""), con=engine)

display(resumo)

amostra = pd.read_sql(text("""
    SELECT ID_Loja, N_Loja, Cidade, Latitude, Longitude
    FROM loja
    ORDER BY ID_Loja
    LIMIT 10
"""), con=engine)

display(amostra)

,total_lojas,lojas_com_coordenadas,lojas_sem_coordenadas
0,80,80.0,0.0


,ID_Loja,N_Loja,Cidade,Latitude,Longitude
0,1,51_ABERDEEN,ABERDEEN,57.14965,-2.09908
1,2,61_AETHELNEY,AETHELNEY,51.05640,-2.93220
2,3,8_ALNERWICK,ALNERWICK,55.41318,-1.70563
3,4,23_ARBINGTON,ARBINGTON,52.24940,0.86460
4,5,2_ASHBORNE,ASHBORNE,53.01670,-1.73330
5,6,29_AYLESBURY,AYLESBURY,51.81665,-0.81458
6,7,21_BALERNO,BALERNO,55.88570,-3.34240
7,8,79_BALLYMENA,BALLYMENA,54.86529,-6.28022
8,9,75_BARNCOMBE,BARNCOMBE,50.91410,0.01870
9,10,56_BEGGAR'S HOLE,BEGGAR'S HOLE,51.51130,-0.87920
